# π0.5-LIBERO × PARC 2026 提出環境

`feature/experiment` の `MyPolicy` をGPUでスモークし、外部通信なしで動く `pi05_submission.zip` を作るノートブックです。既存のSmolVLAノートブックとは環境を分離します。

- ランタイム: GPU（L4 / A100推奨）
- PaliGemmaの利用条件にHugging Face上で同意してください
- このノートブックは既存のLIBERO fine-tuned checkpointの導入用です。追加学習は含みません


In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = "https://github.com/KosukeKomeya/PARC2026_pre.git"
BRANCH = "feature/experiment"
REPO_DIR = Path("/content/PARC2026_pre")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
%cd /content/PARC2026_pre


In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("ColabのランタイムをGPUに変更してください")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("VRAM GiB:", round(props.total_memory / 2**30, 1))
subprocess.run(["nvidia-smi"], check=True)


## Hugging Faceへログイン

`google/paligemma-3b-pt-224` のページで利用条件へ同意してから実行します。トークンをコードやGitHubへ書かないでください。


In [ ]:
%pip install -q "huggingface-hub>=0.34.2,<0.36.0"
from huggingface_hub import notebook_login
notebook_login()


## 1. 固定バージョンの環境と重みを準備

Python 3.10の専用venvを作り、固定コミットのLeRobot/Transformers、固定revisionのπ0.5-LIBERO重み、tokenizerを取得します。初回は時間がかかります。


In [ ]:
!python examples/pi05_parc_colab_setup.py


## 2. 実モデルでスモークテスト

モデルのロード、最初の重い推論、action queueから返す軽い推論を計測します。重い推論が10秒以上なら提出前にGPU・推論step数を見直してください。


In [ ]:
!python examples/pi05_parc_colab_setup.py --skip-download --smoke


## 3. オフライン提出ZIPを作成

チェックポイント、tokenizer、固定ソース、`policy_server.py`、`requirements.txt`をルート直下へまとめます。モデル重みは再圧縮せずZIP64で格納します。


In [ ]:
!python examples/pi05_parc_colab_setup.py --skip-download --build-submission


In [ ]:
from pathlib import Path
from google.colab import files

submission = Path("/content/PARC2026_pre/pi05_submission.zip")
if not submission.is_file():
    raise FileNotFoundError(submission)
print("size GiB:", round(submission.stat().st_size / 2**30, 2))
files.download(str(submission))


## 提出前の最終確認

このノートブックでは静的検査と単体推論を行います。最終的にはPARC配布環境で `python validate_submission.py pi05_submission.zip` と、公開4タスクでの評価を実行してください。成功率だけでなくcollision rateと最大 `/act` レイテンシも確認します。
